In [ ]:
from pathlib import Path
import os
import torch

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data" / "speeches.parquet").is_file()
)
FIG_DIR = PROJECT_ROOT / "results" / "figures"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
source_path = Path(os.environ.get("MORALIZATION_INPUT", PROJECT_ROOT / "data" / "new_prompt_noshot_4omini_analysis_input.parquet"))
if not source_path.is_file():
    raise FileNotFoundError("Full speech text is required: " + str(source_path))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "alexandrainst/scandi-nli-large-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device.type == "cuda" else torch.float32
).to(device)

model.eval()



In [ ]:
# candidate labels

labels_sv = [
    "Makroekonomi och handel",
    "Arbetsmarknad och sysselsättning",
    "Invandring och integration",
    "Klimat, miljö och energi",
    "Hälsa",
    "Social välfärd, bostäder och rättigheter",
    "Utbildning, forskning och kultur",
    "Infrastruktur och jordbruk",
    "Lag, ordning och rättvisa",
    "Försvar och militär",
    "Utrikesfrågor och global handel",
    #"Parlamentariskt förfarande",
]


id2label = model.config.id2label
print(id2label)

entailment_id = [
    i for i, label in id2label.items()
    if "entail" in label.lower()
][0]

print("entailment_id:", entailment_id)

In [ ]:
import numpy as np
from tqdm.auto import tqdm

label_sv_to_en = {
    "Makroekonomi och handel": "Macroeconomics and trade",
    "Arbetsmarknad och sysselsättning": "Labor market and employment",
    "Invandring och integration": "Immigration and integration",
    "Klimat, miljö och energi": "Climate, environment and energy",
    "Hälsa": "Health",
    "Social välfärd, bostäder och rättigheter": "Social welfare, housing and rights",
    "Utbildning, forskning och kultur": "Education, research and culture",
    "Infrastruktur och jordbruk": "Infrastructure and agriculture",
    "Lag, ordning och rättvisa": "Law, order and justice",
    "Försvar och militär": "Defense and military",
    "Utrikesfrågor och global handel": "Foreign affairs and global trade",
   # "Parlamentariskt förfarande": "Parliamentary procedure"
}


def split_text_by_tokens(text, tokenizer, chunk_size=450, stride=50):
    """
    Split text into token-based chunks.
    chunk_size should be safely below model max length because the hypothesis
    and special tokens also need space.
    """
    ids = tokenizer(
        str(text),
        add_special_tokens=False,
        truncation=False,
        verbose=False,
    )["input_ids"]

    if len(ids) == 0:
        return [""]

    chunks = []
    start = 0

    while start < len(ids):
        end = start + chunk_size
        chunk_ids = ids[start:end]

        chunk_text = tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

        chunks.append(chunk_text)

        if end >= len(ids):
            break

        start = max(0, end - stride)

    return chunks

In [ ]:

def softmax_np(x):
    x = x - np.max(x)
    exp_x = np.exp(x)
    return exp_x / exp_x.sum()


@torch.inference_mode()
def classify_speeches_fast_single_label(
    speech_dict,
    labels,
    hypothesis_template="Den här parlamentarisk anförande tillhör kategorin: {}.",
    batch_size=258,
    max_length=512,
    stride=50,
    procedural_threshold=0.9,
):
    """
    Single-label NLI classification.

    This follows Hugging Face zero-shot single-label logic:

        entailment_logits -> softmax across candidate labels

    Returns:
        dict: {speech_id: predicted_label}
    """

    assert len(labels) == 11, f"Expected 11 labels, got {len(labels)}"


    # Pre-tokenize hypotheses once
    hypothesis_ids = [
        tokenizer(
            hypothesis_template.format(label),
            add_special_tokens=False,
            truncation=False,
            verbose=False,
        )["input_ids"]
        for label in labels
    ]

    max_hypothesis_len = max(len(x) for x in hypothesis_ids)
    special_tokens = tokenizer.num_special_tokens_to_add(pair=True)

    max_premise_len = max_length - max_hypothesis_len - special_tokens

    if max_premise_len <= 0:
        raise ValueError("max_length is too small for your labels/template.")

    prediction_dict = {}

    batch_features = []
    batch_meta = []

    # These store entailment logits, not probabilities
    sums = {}
    counts = {}

    def flush_batch():
        nonlocal batch_features, batch_meta

        if not batch_features:
            return

        batch = tokenizer.pad(
            batch_features,
            padding=True,
            return_tensors="pt",
        )

        batch = {k: v.to(device) for k, v in batch.items()}

        logits = model(**batch).logits

        # IMPORTANT:
        # Store entailment logits, not softmax probabilities.
        entailment_logits = logits[:, entailment_id].detach().float().cpu().numpy()

        for score, (speech_id, label_idx) in zip(entailment_logits, batch_meta):
            sums[speech_id][label_idx] += score
            counts[speech_id][label_idx] += 1

        batch_features = []
        batch_meta = []

    for speech_id, text in tqdm(speech_dict.items(), desc="Classifying speeches"):
        text = str(text)

        premise_ids = tokenizer(
            text,
            add_special_tokens=False,
            truncation=False,
            verbose=False,
        )["input_ids"]

        if len(premise_ids) == 0:
            premise_chunks = [[]]
        else:
            premise_chunks = []
            start = 0

            while start < len(premise_ids):
                end = start + max_premise_len
                premise_chunks.append(premise_ids[start:end])

                if end >= len(premise_ids):
                    break

                start = max(0, end - stride)

        sums[speech_id] = np.zeros(len(labels), dtype=np.float32)
        counts[speech_id] = np.zeros(len(labels), dtype=np.float32)

        for premise_chunk_ids in premise_chunks:
            for label_idx, hyp_ids in enumerate(hypothesis_ids):
                input_ids = tokenizer.build_inputs_with_special_tokens(
                    premise_chunk_ids,
                    hyp_ids,
                )

                attention_mask = [1] * len(input_ids)

                feature = {
                    "input_ids": input_ids,
                    "attention_mask": attention_mask,
                }

                token_type_ids = tokenizer.create_token_type_ids_from_sequences(
                    premise_chunk_ids,
                    hyp_ids,
                )

                if len(token_type_ids) == len(input_ids) and any(token_type_ids):
                    feature["token_type_ids"] = token_type_ids

                batch_features.append(feature)
                batch_meta.append((speech_id, label_idx))

                if len(batch_features) >= batch_size:
                    flush_batch()

    flush_batch()

    for speech_id in speech_dict.keys():
        # Average entailment logits across chunks
        avg_logits = sums[speech_id] / np.maximum(counts[speech_id], 1)

        # Single-label zero-shot logic:
        # normalize across the 12 candidate labels
        label_probs = softmax_np(avg_logits)

        best_idx = int(label_probs.argmax())
        best_label_sv = labels[best_idx]

        
        prediction_dict[speech_id] = label_sv_to_en[best_label_sv]
        

    return prediction_dict

In [ ]:
text_col = "text"
id_col = "new_custom_id"



In [ ]:
import pandas as pd

df = pd.read_parquet(source_path)
assert df[id_col].is_unique

In [ ]:
df_sample = df
speech_dict = dict(zip(
    df_sample[id_col].tolist(),
    df_sample[text_col].astype(str).tolist()
))

predictions =  classify_speeches_fast_single_label(
    speech_dict=speech_dict,
    labels=labels_sv,
    batch_size=64,
    max_length=512,
    stride=50,
)
print(type(predictions))


In [ ]:
print(len(predictions))
#for i, v in enumerate(predictions.values()):
#    if v == "Parliamentary procedure":
#        print(i)
        
from collections import Counter

label_distribution = Counter(predictions.values())

label_distribution

In [ ]:
df["topic_label"] = df["new_custom_id"].map(predictions)

In [ ]:
topics = df[["new_custom_id", "topic_label"]].copy()
topics["original_row_idx"] = topics["new_custom_id"].str.extract(r"^row_(\d+)$")[0].astype(int)
topics.to_parquet(PROJECT_ROOT / "data" / "policy_topics.parquet", index=False)
topics["topic_label"].value_counts()
